In [1]:
import os
import re
import glob
import pandas as pd
from difflib import SequenceMatcher
import unicodedata
from pathlib import Path


# CONFIG
PROJECT_ROOT = "/Users/joshmacbook/python_projects/OAD"
RAW_DIR = os.path.join(PROJECT_ROOT, "Data", "Raw")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "Data", "Clean", "Combined")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Data", "Clean", "Combined")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
TOURS = {
    "PGA":  {"folder": "PGA",  "prefix": "PGA"},
    "LIV":  {"folder": "LIV",  "prefix": "liv"},
    "EURO": {"folder": "EURO", "prefix": "EURO"},
}

def extract_year_from_path(path: str) -> int | None:
    fname = os.path.basename(path)
    match = re.search(r"(\d{4})", fname)
    if match:
        return int(match.group(1))
    return None

def load_all_rounds() -> pd.DataFrame:
    dfs = []

    for tour_name, cfg in TOURS.items():
        folder = os.path.join(RAW_DIR, cfg["folder"])
        pattern = os.path.join(folder, f"{cfg['prefix']}_rounds_*.csv")
        files = sorted(glob.glob(pattern))

        if not files:
            print(f"[WARN] No files found for tour {tour_name} using pattern: {pattern}")
            continue

        print(f"[INFO] Loading {len(files)} files for tour {tour_name}")

        for fpath in files:
            year = extract_year_from_path(fpath)
            if year is None:
                print(f"[WARN] Could not extract year from filename: {fpath}")
                continue

            df = pd.read_csv(fpath)
            df["tour"] = tour_name
            df["year"] = year
            dfs.append(df)

    if not dfs:
        print("[ERROR] No data loaded from any tour.")
        return pd.DataFrame()

    all_df = pd.concat(dfs, ignore_index=True)
    print(f"[INFO] Loaded total rows: {len(all_df)}")
    return all_df


def save_by_year(all_df: pd.DataFrame):
    if "year" not in all_df.columns:
        raise KeyError("Combined DataFrame has no 'year' column.")

    years = sorted(all_df["year"].dropna().unique())
    print(f"[INFO] Years found: {years}")

    for yr in years:
        year_df = all_df[all_df["year"] == yr].copy()
        out_path = os.path.join(OUTPUT_DIR, f"combined_rounds_{yr}.csv")
        year_df.to_csv(out_path, index=False)
        print(f"[INFO] Saved {len(year_df)} rows to {out_path}")

if __name__ == "__main__":
    combined_df = load_all_rounds()
    if not combined_df.empty:
        save_by_year(combined_df)
    else:
        print("[ERROR] No combined data to save.")

[INFO] Loading 9 files for tour PGA
[INFO] Loading 4 files for tour LIV
[INFO] Loading 9 files for tour EURO
[INFO] Loaded total rows: 311040
[INFO] Years found: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
[INFO] Saved 34543 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2017.csv
[INFO] Saved 34798 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2018.csv
[INFO] Saved 34064 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2019.csv
[INFO] Saved 26701 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2020.csv
[INFO] Saved 32222 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2021.csv
[INFO] Saved 36234 rows to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2022.csv
[INFO] Saved 37142 rows to 

In [3]:
PROJECT_ROOT = "/Users/joshmacbook/python_projects/OAD"
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Data", "Clean", "Combined")

non_numeric = {"CUT", "WD", "DQ", "MDF", "NAN"}

def clean_finish(val):
    v = str(val).upper().strip()

    if v in non_numeric:
        return v
    if v.startswith("T") and v[1:].isdigit():
        return int(v[1:])
    if v.isdigit():
        return int(v)
    return v

def process_year(year: int):
    fname = f"combined_rounds_{year}.csv"
    path = os.path.join(COMBINED_DIR, fname)

    if not os.path.exists(path):
        print(f"[WARN] File not found for {year}: {path}")
        return

    print(f"[INFO] Processing {path}")
    df = pd.read_csv(path)

    if "fin_text" not in df.columns:
        print(f"[ERROR] 'fin_num' column not found in {path}")
        return

    df["fin_text"] = df["fin_text"].astype(str).str.upper().str.strip()

    df["finish_num"] = df["fin_text"].apply(clean_finish)

    print(df[["fin_text", "finish_num"]].head(10))

    df.to_csv(path, index=False)
    print(f"[INFO] Saved updated file with finish_clean to {path}")

if __name__ == "__main__":
    for yr in range(2017, 2026):
        process_year(yr)

[INFO] Processing /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2017.csv
  fin_text finish_num
0      T12         12
1       20         20
2      T17         17
3      T17         17
4      T17         17
5      T17         17
6      T17         17
7      T17         17
8      T17         17
9      T17         17
[INFO] Saved updated file with finish_clean to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2017.csv
[INFO] Processing /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2018.csv
  fin_text finish_num
0      T22         22
1      T22         22
2      T22         22
3      T22         22
4      T22         22
5      T22         22
6      T22         22
7      T22         22
8      T22         22
9      T22         22
[INFO] Saved updated file with finish_clean to /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2018.csv
[INFO] Processing /Users/joshmacbook/python_projec

In [4]:
BASE_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Combined")
YEARS = [2022, 2023, 2024, 2025]

EVENT_ID_COL = "event_id"
COURSE_COL   = "course_num"

liv_event_id_map_raw = {
    "adelaide":                         1012,
    "andalucia":                        1024,
    "bangkok":                          1006,
    "bedminster":                       1003,
    "boston":                           1004,
    "chicago":                          1005,
    "dallas":                           1031,
    "dallas (team finalstroke play)":   1026,
    "dc":                               1015,
    "greenbrier":                       1017,
    "hong kong":                        1020,
    "houston":                          1022,
    "indianapolis":                     1032,
    "jeddah":                           1007,
    "korea":                            1029,
    "las vegas":                        1019,
    "london":                           1001,
    "mayakoba":                         1009,
    "mexico city":                      1028,
    "miami":                            1021,
    "miami (team finalstroke play)":    1008,
    "michigan (team finalstroke play)": 1033,
    "nashville":                        1023,
    "orlando":                          1011,
    "portland":                         1002,
    "promotions event":                 1018,
    "riyadh":                           1027,
    "singapore":                        1013,
    "tucson":                           1010,
    "tulsa":                            1014,
    "united kingdom":                   1025,
    "valderrama":                       1016,
    "virginia":                         1030,
}

liv_event_id_map = {k.strip().lower(): v for k, v in liv_event_id_map_raw.items()}


def patch_liv_for_year(year: int) -> None:
    path = BASE_DIR / f"combined_rounds_{year}.csv"
    print(f"\n=== {year} ===")
    print(f"Reading: {path}")

    df = pd.read_csv(path)

    required_cols = {"tour", "event_name", EVENT_ID_COL, COURSE_COL}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")

    df["event_name_clean"] = df["event_name"].astype(str).str.strip().str.lower()
    df["tour_clean"] = df["tour"].astype(str).str.strip().str.upper()

    liv_mask = df["tour_clean"].eq("LIV")
    in_map = df["event_name_clean"].isin(liv_event_id_map.keys())
    target_mask = liv_mask & in_map

    print(f"LIV rows: {liv_mask.sum()}")
    print(f"Rows to update: {target_mask.sum()}")

    new_ids = df.loc[target_mask, "event_name_clean"].map(liv_event_id_map).astype(int)

    df.loc[target_mask, EVENT_ID_COL] = new_ids

    df.loc[target_mask, COURSE_COL] = new_ids

    if target_mask.sum() > 0:
        preview = df.loc[target_mask, ["event_name", EVENT_ID_COL, COURSE_COL]].head(10)
        print("Sample updated rows:")
        print(preview)

    df = df.drop(columns=["event_name_clean", "tour_clean"])

    df.to_csv(path, index=False)
    print(f"Updated file written: {path}")


for yr in YEARS:
    patch_liv_for_year(yr)


=== 2022 ===
Reading: /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2022.csv
LIV rows: 1019
Rows to update: 1003
Sample updated rows:
      event_name  event_id  course_num
18486     London      1001        1001
18487     London      1001        1001
18488     London      1001        1001
18489     London      1001        1001
18490     London      1001        1001
18491     London      1001        1001
18492     London      1001        1001
18493     London      1001        1001
18494     London      1001        1001
18495     London      1001        1001
Updated file written: /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2022.csv

=== 2023 ===
Reading: /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2023.csv
LIV rows: 2059
Rows to update: 2011
Sample updated rows:
      event_name  event_id  course_num
18717   Mayakoba      1009        1009
18718   Mayakoba      1009        1009
18719   Mayakoba    

In [5]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Combined")
YEARS = [2022, 2023, 2024, 2025]

EVENT_ID_COL = "event_id"
COURSE_COL   = "course_num"
ROUND_NUM_COL = "round_num"
EVENT_COMPLETED_COL = "event_completed"

liv_event_id_map_raw = {
    "adelaide":                         1012,
    "andalucia":                        1024,
    "bangkok":                          1006,
    "bedminster":                       1003,
    "boston":                           1004,
    "chicago":                          1005,
    "dallas":                           1031,
    "dallas (team finalstroke play)":   1026,
    "dc":                               1015,
    "greenbrier":                       1017,
    "hong kong":                        1020,
    "houston":                          1022,
    "indianapolis":                     1032,
    "jeddah":                           1007,
    "korea":                            1029,
    "las vegas":                        1019,
    "london":                           1001,
    "mayakoba":                         1009,
    "mexico city":                      1028,
    "miami":                            1021,
    "miami (team finalstroke play)":    1008,
    "michigan (team finalstroke play)": 1033,
    "nashville":                        1023,
    "orlando":                          1011,
    "portland":                         1002,
    "promotions event":                 1018,
    "riyadh":                           1027,
    "singapore":                        1013,
    "tucson":                           1010,
    "tulsa":                            1014,
    "united kingdom":                   1025,
    "valderrama":                       1016,
    "virginia":                         1030,
}

liv_event_id_map = {k.strip().lower(): v for k, v in liv_event_id_map_raw.items()}


def patch_and_add_round_date_for_year(year: int) -> None:
    path = BASE_DIR / f"combined_rounds_{year}.csv"
    print(f"\n=== {year} ===")
    print(f"Reading: {path}")

    df = pd.read_csv(path)

    required_cols = {
        "tour",
        "event_name",
        EVENT_ID_COL,
        COURSE_COL,
        ROUND_NUM_COL,
        EVENT_COMPLETED_COL,
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")

    df["event_name_clean"] = df["event_name"].astype(str).str.strip().str.lower()
    df["tour_clean"] = df["tour"].astype(str).str.strip().str.upper()

    liv_mask = df["tour_clean"].eq("LIV")
    in_map = df["event_name_clean"].isin(liv_event_id_map.keys())
    target_mask = liv_mask & in_map

    print(f"LIV rows total:           {liv_mask.sum()}")
    print(f"LIV rows with mapping:    {target_mask.sum()}")

    if target_mask.any():
        new_ids = df.loc[target_mask, "event_name_clean"].map(liv_event_id_map).astype(int)
        df.loc[target_mask, EVENT_ID_COL] = new_ids
        df.loc[target_mask, COURSE_COL] = new_ids

        preview_liv = df.loc[target_mask, ["event_name", EVENT_ID_COL, COURSE_COL]].head(10)
        print("Sample LIV updates:")
        print(preview_liv)

    if "round_date" in df.columns:
        print("Column 'round_date' already exists; it will be overwritten.")

    df["event_completed_dt"] = pd.to_datetime(df[EVENT_COMPLETED_COL])


    group_cols = ["tour", "event_name", "event_completed_dt"]

    max_round_per_event = df.groupby(group_cols)[ROUND_NUM_COL].transform("max")

    offset_days = max_round_per_event - df[ROUND_NUM_COL]

    df["round_date_dt"] = df["event_completed_dt"] - pd.to_timedelta(offset_days, unit="D")
    df["round_date"] = df["round_date_dt"].dt.date.astype(str)

    sample = (
        df.loc[
            df["round_num"].isin([1, 2, 3, 4]),
            ["tour", "event_name", EVENT_COMPLETED_COL, ROUND_NUM_COL, "round_date"],
        ]
        .drop_duplicates()
        .head(20)
    )
    print("Sample round_date assignments:")
    print(sample)

    df = df.drop(columns=["event_name_clean", "tour_clean", "event_completed_dt", "round_date_dt"])

    df.to_csv(path, index=False)
    print(f"Updated file written: {path}")


for yr in YEARS:
    patch_and_add_round_date_for_year(yr)


=== 2022 ===
Reading: /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2022.csv
LIV rows total:           1019
LIV rows with mapping:    1003
Sample LIV updates:
      event_name  event_id  course_num
18486     London      1001        1001
18487     London      1001        1001
18488     London      1001        1001
18489     London      1001        1001
18490     London      1001        1001
18491     London      1001        1001
18492     London      1001        1001
18493     London      1001        1001
18494     London      1001        1001
18495     London      1001        1001
Sample round_date assignments:
     tour                      event_name event_completed  round_num  \
0     PGA  Sentry Tournament of Champions      2022-01-09          1   
1     PGA  Sentry Tournament of Champions      2022-01-09          3   
2     PGA  Sentry Tournament of Champions      2022-01-09          2   
4     PGA  Sentry Tournament of Champions      2022-01-09      

In [6]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Combined")
YEARS = list(range(2018, 2025 + 1))

EVENT_ID_COL = "event_id"   # change if yours is named differently

all_events = []

for year in YEARS:
    path = BASE_DIR / f"combined_rounds_{year}.csv"
    print(f"Reading {path}")

    df = pd.read_csv(path, usecols=["event_name", EVENT_ID_COL])
    df["year"] = year
    all_events.append(df)

# concat
events = pd.concat(all_events, ignore_index=True)

# unique event_name + event_id combos
unique_events = (
    events
    .drop_duplicates(subset=["event_name", EVENT_ID_COL])
    .sort_values(["event_name", EVENT_ID_COL])
    .reset_index(drop=True)
)

print(unique_events)
print(f"\nTotal unique events: {len(unique_events)}")

# optional: save
unique_events.to_csv(
    "/Users/joshmacbook/python_projects/OAD/Data/Clean/unique_event_list_2018_2025.csv",
    index=False
)


Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2018.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2019.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2020.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2021.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2022.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2023.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2024.csv
Reading /Users/joshmacbook/python_projects/OAD/Data/Clean/Combined/combined_rounds_2025.csv
                                          event_name  event_id  year
0                                            3M Open       525  2019
1               A Military Tribute at The Greenbrier       490  2018
2                             ACCIONA Open de España   20

In [7]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Combined")
YEARS = list(range(2017, 2026))

dfs = []

for y in YEARS:
    f = DATA_DIR / f"combined_rounds_{y}.csv"
    df_y = pd.read_csv(f)
    dfs.append(df_y)

combined = pd.concat(dfs, ignore_index=True)

OUT_PATH = DATA_DIR / "combined_rounds_all_2017_2025.csv"
combined.to_csv(OUT_PATH, index=False)

combined.head()

,tour,year,season,event_completed,event_name,event_id,player_name,dg_id,fin_text,round_num,...,prox_fw,great_shots,poor_shots,eagles_or_better,birdies,pars,bogies,doubles_or_worse,finish_num,round_date
0,PGA,2017,2017,2017-01-08,SBS Tournament of Champions,16,"Herman, Jim",12846,T12,1,...,21.601,3.0,2.0,0.0,6.0,12.0,0.0,0.0,12,NaN
1,PGA,2017,2017,2017-01-08,SBS Tournament of Champions,16,"Gomez, Fabian",8571,20,1,...,40.227,4.0,1.0,0.0,5.0,11.0,2.0,0.0,20,NaN
2,PGA,2017,2017,2017-01-08,SBS Tournament of Champions,16,"Knox, Russell",13831,T17,4,...,33.636,1.0,6.0,0.0,5.0,10.0,2.0,1.0,17,NaN
3,PGA,2017,2017,2017-01-08,SBS Tournament of Champions,16,"Knox, Russell",13831,T17,3,...,27.122,1.0,1.0,0.0,7.0,9.0,2.0,0.0,17,NaN
4,PGA,2017,2017,2017-01-08,SBS Tournament of Champions,16,"Knox, Russell",13831,T17,2,...,39.292,2.0,4.0,0.0,6.0,10.0,2.0,0.0,17,NaN


In [14]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310281 entries, 0 to 310280
Data columns (total 37 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   tour              310281 non-null  object 
 1   year              310281 non-null  int64  
 2   season            310281 non-null  int64  
 3   event_completed   310281 non-null  object 
 4   event_name        310281 non-null  object 
 5   event_id          310281 non-null  int64  
 6   player_name       310281 non-null  object 
 7   dg_id             310281 non-null  int64  
 8   fin_text          310281 non-null  object 
 9   round_num         310281 non-null  int64  
 10  course_name       310281 non-null  object 
 11  course_num        310281 non-null  int64  
 12  course_par        305769 non-null  float64
 13  start_hole        297767 non-null  float64
 14  teetime           297767 non-null  object 
 15  round_score       309357 non-null  float64
 16  sg_putt           13